> 在线讲义：[第 24 章 · 策略梯度推导：从似然比技巧到 PPO 和 GRPO](https://wikieden.github.io/math-and-eval-stats/lessons/24-policy-gradient.html) · 本书《数学与实验统计》

# 第 24 章 · 策略梯度推导

讲义：[24-policy-gradient.md](https://wikieden.github.io/math-and-eval-stats/lessons/24-policy-gradient.html)。一格一格往下跑；只算讲义里讲过的例子，练习的数自己算。

## 讲义 ↔ 代码

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 | J、dJ/dθ | `objective / true_gradient` | 0.55、0.125 |
| 第二步 | score | `score` | ±0.5 |
| 第三、四步 | REINFORCE ± 基线 | `reinforce_samples` | 标准差 0.110 / 0.068 |
| 第五步 | PPO clip、GRPO | `ppo_clip_term / grpo_advantages` | +1.20、−0.80；[+1, −1, −1, +1] |

## 定义

In [ ]:
import math
import numpy as np
P_SUCCESS = np.array([0.3, 0.8])

`sigmoid`：

In [ ]:
def sigmoid(x):
    return 1 / (1 + math.exp(-x))

`objective`：第一步：J(θ) = Σ_a π(a)·E[r | a]。

In [ ]:
def objective(theta: float) -> float:
    """第一步：J(θ) = Σ_a π(a)·E[r | a]。"""
    p1 = sigmoid(theta)
    return (1 - p1) * P_SUCCESS[0] + p1 * P_SUCCESS[1]

`true_gradient`：第一步：直接求导 dJ/dθ = σ'(θ)·(0.8 - 0.3)，σ' = σ(1-σ)。

In [ ]:
def true_gradient(theta: float) -> float:
    """第一步：直接求导 dJ/dθ = σ'(θ)·(0.8 - 0.3)，σ' = σ(1-σ)。"""
    p1 = sigmoid(theta)
    return p1 * (1 - p1) * (P_SUCCESS[1] - P_SUCCESS[0])

`score`：第二步：∇θ log π(a)。a=1 时 1-σ(θ)，a=0 时 -σ(θ)。

In [ ]:
def score(theta: float, a: int) -> float:
    """第二步：∇θ log π(a)。a=1 时 1-σ(θ)，a=0 时 -σ(θ)。"""
    p1 = sigmoid(theta)
    return (1 - p1) if a == 1 else -p1

`reinforce_samples`：第三、四步：每个样本的梯度估计 (r - b)·∇log π(a)。

In [ ]:
def reinforce_samples(theta: float, n: int, baseline: float = 0.0, seed: int = 0) -> np.ndarray:
    """第三、四步：每个样本的梯度估计 (r - b)·∇log π(a)。"""
    rng = np.random.default_rng(seed)
    p1 = sigmoid(theta)
    a = (rng.random(n) < p1).astype(int)
    r = (rng.random(n) < P_SUCCESS[a]).astype(float)
    s = np.where(a == 1, 1 - p1, -p1)
    return (r - baseline) * s

`ppo_clip_term`：第五步：PPO 截断目标 min(r·A, clip(r, 1-ε, 1+ε)·A)。

In [ ]:
def ppo_clip_term(ratio: float, adv: float, eps: float = 0.2) -> float:
    """第五步：PPO 截断目标 min(r·A, clip(r, 1-ε, 1+ε)·A)。"""
    return min(ratio * adv, max(min(ratio, 1 + eps), 1 - eps) * adv)

`grpo_advantages`：第五步：GRPO 组内归一化优势 (r - mean) / std。

In [ ]:
def grpo_advantages(rewards) -> np.ndarray:
    """第五步：GRPO 组内归一化优势 (r - mean) / std。"""
    r = np.asarray(rewards, float)
    return (r - r.mean()) / (r.std() + 1e-8)

## 一步一步跑

### 第一步 目标与真梯度

In [ ]:
objective(0.0), true_gradient(0.0)

### 第二步 score function

In [ ]:
score(0.0, 1), score(0.0, 0)

### 第三、四步 有无基线

In [ ]:
[(b, reinforce_samples(0.0, 200_000, baseline=b).reshape(-1, 10).mean(axis=1).std()) for b in (0.0, 0.55)]

### 第五步 PPO 截断与 GRPO

In [ ]:
[ppo_clip_term(r, a) for r, a in [(1.1, 1), (1.5, 1), (0.5, 1), (0.5, -1), (1.5, -1)]], grpo_advantages([1, 0, 0, 1])

## 代码里才看得到的

1. 有无基线用同一组随机数，方差差别只来自基线。
2. std 用总体标准差，加 1e-8 防全对全错除零。
3. 实现里的损失是 −L^CLIP，正负号写反是常见 bug。

## 自己改一改

- 基线改成 0.3，看均值和标准差
- 成功率改成 0.5 和 0.55，看信噪比

In [ ]:
# 在这里改参数试试


## 口诀

> 参数藏在概率里，似然比把梯度搬；
> 对数一拆环境掉，回报加权克隆样；
> 基线期望恒为零，减掉平均方差降；
> PPO 只截得利侧，GRPO 组内比短长。